<a href="https://colab.research.google.com/github/Naman80070/GenAI-GFG/blob/main/GFG_Hugging_Face_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Cell 1 — install libs
!pip install -q transformers[sentencepiece] tqdm

In [3]:
# Cell 2 — imports and helper functions
import pandas as pd
from transformers import pipeline
from tqdm.auto import tqdm
import math

# Utility: load CSV either from drive or upload in Colab
def load_csv(path_or_uploaded=None):
    # If path_or_uploaded is a local path string -> read directly
    # If None -> expect user to use Colab file upload widget (manual)
    if isinstance(path_or_uploaded, str):
        return pd.read_csv(path_or_uploaded)
    raise ValueError("Pass a path (string) to CSV. In Colab you can also upload and pass the file path.")

def save_and_offer_download(df, out_path="tagged_questions.csv"):
    df.to_csv(out_path, index=False)
    print(f"Saved -> {out_path}")
    # In Colab you can then use files.download(out_path) if you want

In [4]:
# # Cell 3 — create classifier (this downloads model; first run may take ~1-2 minutes)
# from transformers import pipeline
# classifier = pipeline(
#     "zero-shot-classification",
#     model="facebook/bart-large-mnli",   # common good default; replace if you prefer a smaller/faster model
#     device=0  # set to -1 for CPU; in Colab set to 0 if GPU runtime enabled
# )

In [5]:
from transformers import pipeline
classifier = pipeline(
    "zero-shot-classification",
    model="valhalla/distilbart-mnli-12-3",
    device=0
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/283 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

In [6]:
# Cell 4 — define main labels and detailed-label map
# Edit these lists to suit your domain and company-specific tags.
MAIN_LABELS = [
    "Python", "SQL", "Excel", "Behavioural", "HR", "Data Structures",
    "Algorithms", "System Design", "Machine Learning", "DevOps",
    "Linux", "JavaScript", "HTML/CSS", "Django", "React", "Aptitude",
    "Testing", "Regex", "Other"
]

# For each main label, list candidate detailed labels (examples). Expand as needed.
DETAILED_LABELS = {
    "Python": ["List", "Dictionary", "Tuple", "List Comprehension", "Generators", "Function", "Decorator", "OOP", "File I/O", "Pandas", "Iterators", "Exception Handling"],
    "SQL": ["SELECT", "JOIN", "GROUP BY", "ORDER BY", "Window Function", "Indexes", "Subquery", "Normalization", "Transactions"],
    "Excel": ["Pivot Table", "VLOOKUP", "INDEX MATCH", "Charts", "Formulas", "Macros", "Conditional Formatting"],
    "Behavioural": ["STAR", "Teamwork", "Leadership", "Conflict", "Strengths Weaknesses", "Work Ethic"],
    "HR": ["Notice Period", "Salary Expectation", "Company Policy", "Background Check"],
    "Data Structures": ["Array", "Linked List", "Stack", "Queue", "Tree", "Graph", "Hash Table", "Heap"],
    "Algorithms": ["Sorting", "Searching", "Dynamic Programming", "Greedy", "Backtracking", "Recursion"],
    "Machine Learning": ["Regression", "Classification", "Clustering", "Feature Engineering", "Model Evaluation", "Neural Networks"],
    "DevOps": ["CI/CD", "Docker", "Kubernetes", "Monitoring", "Infrastructure as Code"],
    "Linux": ["Shell", "Permissions", "Processes", "Networking", "Systemctl"],
    "JavaScript": ["Closure", "Promises", "Async/Await", "DOM", "Event Loop"],
    "HTML/CSS": ["Flexbox", "Grid", "Selectors", "Responsive Design"],
    "Django": ["ORM", "Views", "Templates", "Middleware", "Authentication"],
    "React": ["Hooks", "State", "Props", "Lifecycle", "Redux"],
    "Aptitude": ["Quantitative", "Logical Reasoning", "Puzzles"],
    "Testing": ["Unit Testing", "Integration Testing", "Test Automation", "TDD"],
    "Regex": ["Pattern Matching", "Groups", "Lookahead", "Lookbehind"],
    "Other": ["Misc"]
}

In [5]:
# # Cell 5 — main classification + detailed classification (batching)
# def classify_questions(df,
#                        text_column="question",
#                        classifier=classifier,
#                        main_labels=MAIN_LABELS,
#                        detailed_labels_map=DETAILED_LABELS,
#                        batch_size=16,
#                        hypothesis_template="This question is about {}."):
#     texts = df[text_column].astype(str).tolist()
#     n = len(texts)
#     top_tags = []
#     top_scores = []
#     detailed_tags = []
#     detailed_scores = []

#     # Stage 1: predict main label (single best)
#     for i in tqdm(range(0, n, batch_size), desc="Main-label batches"):
#         batch_texts = texts[i:i+batch_size]
#         results = classifier(
#             batch_texts,
#             candidate_labels=main_labels,
#             hypothesis_template=hypothesis_template,
#             multi_label=False   # single best label (top-1)
#         )
#         # pipeline returns list of dicts if input is list
#         for r in results:
#             label = r['labels'][0]
#             score = float(r['scores'][0])
#             top_tags.append(label)
#             top_scores.append(score)

#     # Stage 2: predict detailed label from the subset for the predicted main
#     # We'll run classifier per-row but only over the smaller candidate set
#     for idx, text in enumerate(tqdm(texts, desc="Detailed per-row")):
#         main_tag = top_tags[idx]
#         cand = detailed_labels_map.get(main_tag, ["Misc"])
#         # If candidate list is tiny, still works.
#         res = classifier(
#             text,
#             candidate_labels=cand,
#             hypothesis_template="This question is about {}.",
#             multi_label=False
#         )
#         dlabel = res['labels'][0]
#         dscore = float(res['scores'][0])
#         detailed_tags.append(dlabel)
#         detailed_scores.append(dscore)

#     # Attach to dataframe
#     df = df.copy()
#     df["Tag"] = top_tags
#     df["Tag_confidence"] = top_scores
#     df["Detailed Tag"] = detailed_tags
#     df["Detailed_confidence"] = detailed_scores
#     return df

In [10]:
def classify_questions(df, text_column="question", batch_size=64):
    texts = df[text_column].astype(str).tolist()

    # ---------- MAIN TAG (BATCHED) ----------
    main_results = classifier(
        texts,
        candidate_labels=MAIN_LABELS,
        batch_size=batch_size,
        multi_label=False
    )

    tags = [r["labels"][0] for r in main_results]
    tag_scores = [float(r["scores"][0]) for r in main_results]

    df = df.copy()
    df["Tag"] = tags
    df["Tag_confidence"] = tag_scores

    detailed_tags = [""] * len(df)
    detailed_scores = [0.0] * len(df)

    # ---------- DETAILED TAG (GROUPED BATCHING) ----------
    for main_label in set(tags):

        indices = df[df["Tag"] == main_label].index.tolist()
        if not indices:
            continue

        candidate_labels = DETAILED_LABELS.get(main_label, ["Misc"])

        group_texts = df.loc[indices, text_column].astype(str).tolist()

        results = classifier(
            group_texts,
            candidate_labels=candidate_labels,
            batch_size=batch_size,
            multi_label=False
        )

        for idx, res in zip(indices, results):
            detailed_tags[idx] = res["labels"][0]
            detailed_scores[idx] = float(res["scores"][0])

    df["Detailed Tag"] = detailed_tags
    df["Detailed_confidence"] = detailed_scores

    return df

In [11]:
# Cell 6 — example: load CSV, run, save
# Replace 'your_file.csv' with the path (for Colab you can upload the CSV and provide its path)
csv_path = "/content/sample_data/ai.csv"   # <-- change this to your file path in Colab
df = pd.read_csv(csv_path)            # assumes a column named 'question' exists; if not replace name

# If your column is named differently, set text_column accordingly in classify_questions
tagged_df = classify_questions(df, text_column="question", batch_size=8)

# Save results
out_path = "/content/sample_data/Naman-ai.csv"
tagged_df.to_csv(out_path, index=False)
print("Saved:", out_path)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Saved: /content/sample_data/Naman-ai.csv


In [12]:
# Optional: show a few examples
tagged_df.head(20)

,question,company,rating,position,date,comments,Tag,Tag_confidence,Detailed Tag,Detailed_confidence
0,Basic algos questions,Interviewed at Google,4.3,AI Researcher,29 Mar 2019,3,Algorithms,0.480200,Backtracking,0.335995
1,4 16 64 256 solve using for loop.,Interviewed at Trutech Web Solutions,2.9,Java Developer AI,1 Dec 2015,3,JavaScript,0.158718,Closure,0.625130
2,Tell us about your projects and resume,Interviewed at Pirimid Fintech,4.7,AI/ML Engineer,9 Jul 2021,2,Other,0.132483,Misc,0.026095
3,Assignment was building an algorithm to play t...,Interviewed at NewsBytes,3.3,AI Engineer,6 May 2024,2,Algorithms,0.140506,Dynamic Programming,0.244768
4,How to make comments classification?,Interviewed at zarplata.ru,5.0,AI/ML Engineer,27 Oct 2021,1,Other,0.175971,Misc,0.025142
5,They asked me which algo will use use if you w...,Interviewed at WB Hotels & Resorts.,-0.1,AI Ml,15 Mar 2022,1,Testing,0.240153,Test Automation,0.418235
6,The interviewer asked me to sort a dictionary ...,Interviewed at E42.ai,4.5,AI Platfrom Developer,2 Aug 2022,1,Python,0.752145,Dictionary,0.603556
7,What was your previous expereince,Interviewed at INOYAD Technologies,-0.1,AI/ML Engineer,31 Jul 2023,1,Other,0.187505,Misc,0.002801
8,The main thing they focused on the major proje...,Interviewed at MKCL,3.6,AI/ML Development,10 Jun 2021,1,Linux,0.127485,Networking,0.247226
9,The company is apparently building software fo...,Interviewed at Scotty Labs,5.0,AI Software,18 Apr 2018,1,Other,0.219561,Misc,0.348849


In [1]:
# This checks if you are using GPU or CPU

import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
GPU name: Tesla T4
